# Topic 2: Data Quality Framework

> **Phase 7 → Week 15 → Topic 2**

---

## What Is Data Quality?

Data quality (DQ) is confidence that your data is correct enough to make decisions from. Five dimensions:

| Dimension | Question | Example check |
|-----------|----------|---------------|
| Completeness | Are all expected records present? | Row count in expected range |
| Validity | Do values conform to rules? | `amount > 0`, `status IN (...)` |
| Uniqueness | No unexpected duplicates? | `order_id` has no duplicates |
| Consistency | Do related values agree? | `customer_id` exists in customers table |
| Timeliness | Is data fresh enough? | `MAX(event_time) < 2h ago` |

---

## Interview Cheat Sheet

**Q: How do you implement data quality checks in a Spark pipeline?**
> Three options: (1) Custom checks — write Spark aggregations that compute null rates, value ranges, row counts; compare to thresholds; raise exceptions or route bad data to a dead-letter table. (2) Great Expectations — open-source DQ framework with 300+ built-in checks, HTML reports, checkpoint integration. (3) Delta Live Tables expectations — declarative `@dlt.expect` in Databricks pipelines. In production: run DQ after Silver write, fail the pipeline or alert if critical rules are violated.

**Q: What is a dead-letter table and why is it better than just dropping bad records?**
> A dead-letter table stores records that failed DQ checks — instead of silently discarding them. Benefits: (1) you can investigate why records are bad (upstream schema change? data corruption?), (2) you can replay them after fixing the issue, (3) you have an audit trail, (4) QA can verify that bad records are genuinely bad. Always quarantine, never discard.

**Q: What is referential integrity and how do you check it in Spark?**
> Referential integrity: every foreign key value must exist in the referenced table. Example: every `customer_id` in the orders table must exist in the customers table. Check with a left anti-join: `orders.join(customers, on='customer_id', how='left_anti')` returns orders with no matching customer. Count the result — if > 0, referential integrity is violated.

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from enum import Enum

spark = SparkSession.builder \
    .appName("Week15 - Data Quality Framework") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Data Quality Framework — ready")

## Part 1: Custom DQ Check Framework

In [ ]:
class Severity(Enum):
    WARN  = "WARN"   # log and continue
    ERROR = "ERROR"  # fail the pipeline

@dataclass
class DQResult:
    rule:     str
    passed:   bool
    severity: Severity
    detail:   str

class DataQualityChecker:
    def __init__(self, df: DataFrame, table_name: str):
        self.df    = df
        self.name  = table_name
        self.total = df.count()
        self._results: List[DQResult] = []

    def check_not_null(self, col: str, severity=Severity.ERROR) -> 'DataQualityChecker':
        null_count = self.df.filter(F.col(col).isNull()).count()
        null_rate  = null_count / self.total if self.total else 0
        passed = null_count == 0
        self._results.append(DQResult(
            rule=f"{col} IS NOT NULL", passed=passed, severity=severity,
            detail=f"{null_count} nulls ({null_rate:.1%})"
        ))
        return self

    def check_null_rate(self, col: str, max_rate: float, severity=Severity.WARN) -> 'DataQualityChecker':
        null_count = self.df.filter(F.col(col).isNull()).count()
        null_rate  = null_count / self.total if self.total else 0
        passed = null_rate <= max_rate
        self._results.append(DQResult(
            rule=f"{col} null_rate <= {max_rate:.1%}", passed=passed, severity=severity,
            detail=f"actual={null_rate:.2%}"
        ))
        return self

    def check_unique(self, col: str, severity=Severity.ERROR) -> 'DataQualityChecker':
        dupes = self.total - self.df.select(col).distinct().count()
        self._results.append(DQResult(
            rule=f"{col} is unique", passed=(dupes == 0), severity=severity,
            detail=f"{dupes} duplicates"
        ))
        return self

    def check_value_in(self, col: str, allowed: List, severity=Severity.ERROR) -> 'DataQualityChecker':
        bad = self.df.filter(~F.col(col).isin(allowed) | F.col(col).isNull()).count()
        self._results.append(DQResult(
            rule=f"{col} IN {allowed}", passed=(bad == 0), severity=severity,
            detail=f"{bad} invalid values"
        ))
        return self

    def check_row_count(self, min_rows: int, max_rows: Optional[int]=None, severity=Severity.ERROR) -> 'DataQualityChecker':
        bound = f">={min_rows}" + (f" and <={max_rows}" if max_rows else "")
        passed = self.total >= min_rows and (max_rows is None or self.total <= max_rows)
        self._results.append(DQResult(
            rule=f"row_count {bound}", passed=passed, severity=severity,
            detail=f"actual={self.total}"
        ))
        return self

    def check_value_range(self, col: str, min_val=None, max_val=None, severity=Severity.ERROR) -> 'DataQualityChecker':
        cond = F.lit(True)
        if min_val is not None: cond = cond & (F.col(col) >= min_val)
        if max_val is not None: cond = cond & (F.col(col) <= max_val)
        bad = self.df.filter(~cond).count()
        rule = f"{col} in [{min_val}, {max_val}]"
        self._results.append(DQResult(rule=rule, passed=(bad == 0), severity=severity, detail=f"{bad} out-of-range"))
        return self

    def check_referential_integrity(self, col: str, ref_df: DataFrame, ref_col: str, severity=Severity.ERROR) -> 'DataQualityChecker':
        orphans = self.df.join(ref_df.select(ref_col), self.df[col] == ref_df[ref_col], "left_anti").count()
        self._results.append(DQResult(
            rule=f"{col} references {ref_col}", passed=(orphans == 0), severity=severity,
            detail=f"{orphans} orphan records"
        ))
        return self

    def report(self, raise_on_error: bool = True) -> List[DQResult]:
        print(f"\nDQ Report — {self.name} ({self.total:,} rows)")
        print("─" * 60)
        errors = []
        for r in self._results:
            icon = "✅" if r.passed else ("❌" if r.severity == Severity.ERROR else "⚠️")
            print(f"  {icon} [{r.severity.value}] {r.rule}: {r.detail}")
            if not r.passed and r.severity == Severity.ERROR:
                errors.append(r)
        print()
        if errors and raise_on_error:
            raise ValueError(f"{len(errors)} DQ ERROR(s) — pipeline halted")
        return self._results

print("DataQualityChecker defined")

## Part 2: Full DQ Pipeline Demo

In [ ]:
# Test data
orders = spark.createDataFrame([
    ("O001", "C001",  250.0, "shipped"),
    ("O002", None,     80.0, "pending"),   # null customer
    ("O003", "C999",  180.0, "shipped"),   # orphan customer
    ("O004", "C002", -10.0,  "shipped"),   # negative amount
    ("O005", "C001",  320.0, "unknown"),   # invalid status
    ("O001", "C001",  250.0, "shipped"),   # duplicate order_id
], ["order_id", "customer_id", "amount", "status"])

customers = spark.createDataFrame([
    ("C001", "Alice"),
    ("C002", "Bob"),
], ["customer_id", "name"])

# Run DQ checks
try:
    (
        DataQualityChecker(orders, "silver.orders")
        .check_row_count(min_rows=1)
        .check_not_null("customer_id", severity=Severity.ERROR)
        .check_unique("order_id", severity=Severity.ERROR)
        .check_value_range("amount", min_val=0, severity=Severity.ERROR)
        .check_value_in("status", ["shipped", "pending", "cancelled"], severity=Severity.ERROR)
        .check_referential_integrity("customer_id", customers, "customer_id", severity=Severity.WARN)
        .report(raise_on_error=True)
    )
except ValueError as e:
    print(f"Pipeline halted: {e}")

## Part 3: Dead Letter Pattern

In [ ]:
def split_valid_invalid(
    df: DataFrame,
    conditions: Dict[str, Any]
) -> tuple:
    """
    Split DataFrame into (valid, invalid) based on conditions.
    Invalid rows get a _rejection_reason column.
    Returns (valid_df, dead_letter_df)
    """
    from functools import reduce
    import operator

    # Tag each row with its failure reasons
    df_tagged = df

    reason_col = F.lit("")
    for rule_name, condition in conditions.items():
        reason_col = F.when(
            ~condition,
            F.concat(reason_col, F.lit(f"|{rule_name}"))
        ).otherwise(reason_col)

    df_tagged = df.withColumn("_rejection_reason", reason_col)

    valid       = df_tagged.filter(F.col("_rejection_reason") == "").drop("_rejection_reason")
    dead_letter = df_tagged.filter(F.col("_rejection_reason") != "") \
                           .withColumn("_rejected_at", F.current_timestamp())

    return valid, dead_letter


# Apply to orders data
conditions = {
    "customer_not_null": F.col("customer_id").isNotNull(),
    "amount_positive":   F.col("amount") > 0,
    "valid_status":      F.col("status").isin("shipped", "pending", "cancelled"),
}

valid_orders, dead_letter = split_valid_invalid(orders, conditions)

print(f"Valid rows:       {valid_orders.count()}")
print(f"Dead letter rows: {dead_letter.count()}")
print("\nDead letter (rejected records + reason):")
dead_letter.select("order_id", "customer_id", "amount", "status", "_rejection_reason").show(truncate=False)

# In production: write valid_orders to Silver, dead_letter to s3://bucket/dead-letter/silver/orders/date=.../
print("In production:")
print("  valid_orders.write.mode('overwrite').parquet('s3://bucket/silver/orders/')")
print("  dead_letter.write.mode('append').parquet('s3://bucket/dead-letter/orders/')")

## Part 4: Statistical Anomaly Detection

In [ ]:
# Distribution shift detection — compare today vs 7-day baseline
import json, os, shutil

STATS_PATH = "/tmp/dq_stats"
if os.path.exists(STATS_PATH): shutil.rmtree(STATS_PATH)
os.makedirs(STATS_PATH)

def compute_column_stats(df: DataFrame, col: str) -> Dict:
    row = df.select(
        F.count(col).alias("count"),
        F.avg(col).alias("mean"),
        F.stddev(col).alias("stddev"),
        F.percentile_approx(col, 0.5).alias("p50"),
        F.percentile_approx(col, 0.95).alias("p95"),
        F.min(col).alias("min"),
        F.max(col).alias("max"),
    ).collect()[0]
    return {k: (float(row[k]) if row[k] is not None else None) for k in row.asDict()}

def check_distribution_shift(
    current_stats: Dict,
    baseline_stats: Dict,
    col: str,
    mean_tolerance: float = 0.2,   # 20% change in mean
    count_tolerance: float = 0.3   # 30% change in count
):
    violations = []
    if baseline_stats.get("mean") and current_stats.get("mean"):
        mean_shift = abs(current_stats["mean"] - baseline_stats["mean"]) / baseline_stats["mean"]
        if mean_shift > mean_tolerance:
            violations.append(f"{col} mean shifted {mean_shift:.0%} (tolerance={mean_tolerance:.0%})")
    if baseline_stats.get("count") and current_stats.get("count"):
        count_shift = abs(current_stats["count"] - baseline_stats["count"]) / baseline_stats["count"]
        if count_shift > count_tolerance:
            violations.append(f"{col} count shifted {count_shift:.0%} (tolerance={count_tolerance:.0%})")
    return violations

# Simulate today vs baseline
today = spark.createDataFrame(
    [(float(i), "C001") for i in range(1, 101)],
    ["amount", "customer_id"]
)
# Simulate a distribution shift: today's amounts are 50% higher
today_shifted = today.withColumn("amount", F.col("amount") * 1.5)

baseline_stats = compute_column_stats(today, "amount")
current_stats  = compute_column_stats(today_shifted, "amount")

print(f"Baseline mean: {baseline_stats['mean']:.1f}")
print(f"Current mean:  {current_stats['mean']:.1f}")

violations = check_distribution_shift(current_stats, baseline_stats, "amount")
if violations:
    print("\n⚠️  Distribution shift detected:")
    for v in violations:
        print(f"  {v}")
else:
    print("\n✅ Distribution within tolerance")

## Exercises

1. Extend `DataQualityChecker` with two more checks: `check_no_future_dates(col)` (no timestamps more than 1 hour in the future) and `check_distribution_shift(col, baseline_df, mean_tolerance=0.2)`.
2. Implement the full dead-letter pipeline: route invalid orders to `s3://bucket/dead-letter/orders/date=YYYY-MM-DD/`, valid orders to Silver. Make the `_rejection_reason` column contain ALL failing rule names (pipe-separated) for rows with multiple violations.
3. Write a DQ report that saves metrics as JSON to S3 after each pipeline run. Then write a second job that reads the last 7 days of DQ reports and flags any metric that has been steadily worsening (monotonic increase in null_rate over 7 days).
4. You receive a dataset where 5% of `customer_id` values are orphans (no match in customers table). Do you: (a) fail the pipeline, (b) warn and route to dead-letter, or (c) warn and continue? Justify your answer based on business context.
5. What is the difference between data quality checks run AT WRITE TIME (inline in the pipeline) vs AFTER WRITE (separate DQ job)? Give a production scenario where each approach is better.